# From Java to C

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Read input with scanf (each variable with its &) and print with printf, using widths and precision such as %7.2f.
- Choose between char, int, unsigned and double, and predict integer division, char arithmetic and unsigned wrap-around.
- Write if, loops and switch the C way, with break and deliberate fall-through.

**In this notebook**, one idea at a time (run each cell before moving on):
1. The shape of a C program
2. From source code to a running program
3. Types: no String, no boolean
4. printf: printing with a format
5. scanf and the &
6. Arithmetic, and chars as numbers
7. if, loops and switch: almost like Java
8. unsigned and wrap-around
9. Common mistakes

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. The shape of a C program

A C program has no class around it: it is a set of **functions**, and it starts at `main`. `#include <stdio.h>` brings in the standard input/output functions, such as `printf`. `main` returns an `int` to the operating system: `0` means "everything went fine".

```c
#include <stdio.h>

int main(void) {
    printf("Hello!\n");
    return 0;
}
```

`printf` doesn't end the line by itself: `\n` inside the text is the line break.

In [ ]:
#include <stdio.h>

int main(void) {
    printf("Hello from C!\n");
    printf("Same ");
    printf("line\n");
    return 0;
}

🧪 **Try it:** remove the `\n` from the first `printf` and run. In the next cells, plain statements are run inside a `main` for you, with the usual `#include`s.

## 2. From source code to a running program

Java compiles to bytecode that the JVM runs. C is compiled straight to **machine code** for your processor: the compiler (`gcc` or `clang`) turns `hello.c` into a program the CPU runs directly. Sphinx compiles each cell this way, with warnings on: a **warning** is the compiler saying "this compiles, but it's probably a bug". Treat warnings as errors.

#### 🎞️ From source code to a running C program

![hello.c is preprocessed, compiled and linked by gcc into machine code, which the CPU runs directly, unlike Java's bytecode on the JVM](compile-steps.svg)

In [ ]:
int apples = 3;
int pears = 4;
printf("%d fruits\n", apples + pears);

🧪 **Try it:** delete a `;` and read the compiler's error. Then add a line `int unused = 1;` and read the **warning**: the program still runs, but Sphinx points it out.

## 3. Types: no String, no boolean

C has the number types you know (`int`, `long`, `double`, `char`), but **no `String`** (text is an array of `char`, in a later unit) and, by default, **no `boolean`**: a condition is an `int`, where `0` is false and anything else is true. `#include <stdbool.h>` adds `bool`, `true` and `false` as names for those.

The size of each type depends on the machine: `sizeof` tells you how many bytes it takes.

In [ ]:
printf("char: %zu byte\n", sizeof(char));
printf("int: %zu bytes\n", sizeof(int));
printf("long: %zu bytes\n", sizeof(long));
printf("double: %zu bytes\n", sizeof(double));

int adult = 20 >= 18;          // a comparison gives 1 or 0
printf("20 >= 18 gives %d, 5 > 9 gives %d\n", adult, 5 > 9);
if (42) {
    printf("any non-zero value counts as true\n");
}

🧪 **Try it:** change `if (42)` to `if (0)` and to `if (-1)`. Which ones print?

## 4. printf: printing with a format

`printf` takes a **format string** with placeholders, then one value per placeholder, in order. The placeholder must match the value's type:

| Placeholder | For | Example |
|---|---|---|
| `%d` | `int` | `printf("%d", 42)` |
| `%ld` | `long` | `printf("%ld", 5000000000L)` |
| `%f`, `%.2f` | `double` (2 decimals) | `printf("%.2f", 3.14159)` → `3.14` |
| `%c` | `char` | `printf("%c", 'A')` |
| `%s` | text | `printf("%s", "Ada")` |
| `%zu` | a `sizeof` | `printf("%zu", sizeof(int))` |

A number between `%` and the letter sets the **width**: `%5d` right-aligns in 5 columns, `%-5d` left-aligns, `%7.2f` uses 7 columns with 2 decimals.

In [ ]:
int qty = 3;
double price = 4.5;
char size = 'M';
printf("%d shirts, size %c, at %.2f each\n", qty, size, price);
printf("[%5d]\n[%-5d]\n", qty, qty);
printf("[%7.2f]\n", price * qty);
printf("%s has %d letters\n", "Ada", 3);

🧪 **Try it:** print `price` with `%d` and read the warning: the placeholder doesn't match the type, and the output is garbage. Then build a small table with `%-10s %5d`.

## 5. scanf and the &

`scanf` reads input with the same placeholders (but `%lf` for a `double`). It must **write into** your variables, so it gets their **addresses**: `&qty` is "where `qty` lives". It returns how many values it managed to read, which tells you whether the input was valid.

Notebook cells can't read the keyboard, so below `sscanf` reads from a text instead; it works exactly like `scanf`.

In [ ]:
int qty;
double price;
int read = sscanf("3 2.50", "%d %lf", &qty, &price);   // like scanf("%d %lf", &qty, &price)
printf("read %d values: %d x %.2f = %.2f\n", read, qty, price, qty * price);

int n = 0;
int ok = sscanf("abc", "%d", &n);
printf("reading \"abc\" as a number read %d values\n", ok);

🧪 **Try it:** remove the `&` before `qty` and read the warning. In a real program that `scanf` would write to a random place in memory. Put it back.

## 6. Arithmetic, and chars as numbers

Arithmetic works as in Java: `int / int` drops the decimals (`7 / 2` is `3`), `%` is the remainder, and `(double)` converts. A `char` is a small number, the character's code: `'A' + 2` is the code of `'C'`, and `c - '0'` turns the digit character `'7'` into the number 7. Print it with `%c` to see the character, or `%d` to see the code.

In [ ]:
printf("%d %d %.1f\n", 7 / 2, 7 % 2, (double) 7 / 2);
char c = 'A';
printf("%c has code %d\n", c, c);
printf("%c\n", c + 2);
char digit = '7';
printf("as a number: %d\n", digit - '0');

🧪 **Try it:** print the letters from `'a'` to `'z'` with a `for (char ch = 'a'; ch <= 'z'; ch++)` loop. What is `'a' - 'A'`?

### ✏️ Exercise: Is it upper case?

Return 1 when the char is a letter from A to Z, and 0 otherwise. C has no boolean by default: 1 is true and 0 is false. A char is a small number, so you can compare it with `'A'` and `'Z'`.

In [ ]:
int is_upper(char c) {
    // TODO: compare c with the range
    return 0;
}

int main(void) {
    check_int("'A'", is_upper('A'), 1);
    check_int("'z'", is_upper('z'), 0);
    check_int("'5'", is_upper('5'), 0);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int is_upper(char c) {
    return c >= 'A' && c <= 'Z';
}

int main(void) {
    check_int("'A'", is_upper('A'), 1);
    check_int("'z'", is_upper('z'), 0);
    check_int("'5'", is_upper('5'), 0);
    return 0;
}
```

</details>

## 7. if, loops and switch: almost like Java

`if`/`else`, `for`, `while`, `do … while` and `switch` look just like Java. C only has the classic `switch`, with `case …:` (no `->` arrows), and as in Java's classic form, a `case` with no `break` **falls through** into the next one. Sometimes that's useful (several cases sharing code), but forgetting a `break` is a classic bug.

In [ ]:
for (int qty = 1; qty <= 4; qty++) {
    printf("%d: ", qty);
    switch (qty) {
        case 1:
            printf("single\n");
            break;
        case 2:
            printf("pair\n");
            break;
        default:
            printf("many\n");
    }
}

🧪 **Try it:** delete the first `break` and run: what does qty 1 print now? The compiler may warn about the fall-through.

### ✏️ Exercise: Sum of the digits

Add up the digits of a non-negative int: 1234 gives 10. The loop looks just like Java's.

In [ ]:
int digit_sum(int n) {
    // TODO: take n % 10, then divide by 10
    return 0;
}

int main(void) {
    check_int("1234", digit_sum(1234), 10);
    check_int("0", digit_sum(0), 0);
    check_int("909", digit_sum(909), 18);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int digit_sum(int n) {
    int sum = 0;
    while (n > 0) {
        sum += n % 10;
        n /= 10;
    }
    return sum;
}

int main(void) {
    check_int("1234", digit_sum(1234), 10);
    check_int("0", digit_sum(0), 0);
    check_int("909", digit_sum(909), 18);
    return 0;
}
```

</details>

## 8. unsigned and wrap-around

An `unsigned` type holds no negative numbers, only 0 up to its maximum. Going past the maximum **wraps around** to 0, like a car's odometer, and going below 0 wraps to the maximum. An `unsigned char` (1 byte) goes from 0 to 255. Signed overflow, on the other hand, is undefined behavior in C: avoid it.

In [ ]:
unsigned char counter = 254;
for (int i = 0; i < 4; i++) {
    printf("%u ", (unsigned) counter);
    counter++;
}
printf("\n");
unsigned int zero = 0;
printf("0 - 1 as unsigned: %u\n", zero - 1);

🧪 **Try it:** start the counter at 0 and count **down** 3 times with `counter--`. Where does it go?

## 9. Common mistakes

- **Forgetting `&` in `scanf`:** it compiles (maybe with a warning) and writes somewhere it shouldn't.
- **`=` instead of `==` in a condition:** in C it **compiles**, because an assignment gives a number. `if (x = 5)` sets x to 5 and is always true. With warnings on, the compiler points it out.
- **A placeholder that doesn't match the type:** `%d` for a `double` prints garbage.
- **A missing `break`** in a `switch`.
- **Integer division:** `1 / 2` is `0`.

In [ ]:
int x = 3;
if (x == 5) {
    printf("five\n");
} else {
    printf("x is %d\n", x);
}

🧪 **Try it:** change `x == 5` to `x = 5`, run, and read the warning: what is printed, and what is x now?

## In short

C has no classes and no `String` or `boolean` built in: a program is functions, starting at `int main(void)`. `#include <stdio.h>` brings in `scanf` and `printf`. A format string says what to read or print: `%d` an `int`, `%f` a `double` (`%lf` in `scanf`), `%c` a `char`.

```c
#include <stdio.h>

int main(void) {
    int qty;
    double price;
    if (scanf("%d %lf", &qty, &price) != 2) {   // how many values it read
        printf("invalid input\n");
        return 1;
    }
    printf("%3d x %7.2f\n", qty, price);   // width 3; width 7 with 2 decimals
    switch (qty) {
        case 1:  printf("single\n"); break;
        case 2:  printf("pair\n");   break;
        default: printf("many\n");
    }
    return 0;
}
```

A `char` is a small number (`'A' + 2` is `'C'`), `7 / 2` is `3`, and an `unsigned` value wraps around past its maximum. A `case` without `break` falls into the next one.

**Watch out:** `scanf` needs the address of each variable: `scanf("%d", &qty)`. Without `&` the program compiles, then writes somewhere it shouldn't.

## Readings

- [Lecture 1: C](https://cs50.harvard.edu/x/2026/notes/1/) (CS50x (Harvard), lecture notes, 25 min): From "Hello, World" to "Types", "Format Codes" and "Loops and meow.c". CS50 uses its own library: get_int and string come from cs50.h, which you don't have here. Read input with scanf instead, as in the example above.
- [Input and output](https://www.freecodecamp.org/news/the-c-beginners-handbook/#heading-input-and-output) (freeCodeCamp, The C Beginner's Handbook, 6 min): Optional: the printf format specifiers and scanf with &. "Unsigned integers" and "The problem with overflow", earlier in the same handbook, explain the wrap-around.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Receipt Line** · Easy
- **Character Codes** · Easy
- **Odometer** · Medium
- **Day of the Year** · Medium
- **Command-Line Calculator** · Hard